In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd
import numpy as np

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

import napari

import matplotlib.pyplot as plt
import matplotlib

from tqdm import tqdm

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# Provide path to input files

img_denoised_dir = Path(r'input_path')
img_denoised_path = os.path.join(img_denoised_dir,'*.stk') 
img_denoised_files = np.sort(glob.glob(img_denoised_path))

img_raw_dir = Path(r'input_path')
img_raw_path = os.path.join(img_raw_dir,'*.stk') 
img_raw_files = np.sort(glob.glob(img_raw_path))

print(len(img_denoised_files))
print(len(img_raw_files))

In [ ]:
# Provide path to output directory

img_composite_dir = r'output_dir'

In [ ]:
# Read images into list

img_denoised = []
img_raw = []

for file in tqdm(img_denoised_files):
    image = imread(file)
    img_denoised.append(image)

for file in tqdm(img_raw_files):
    image = imread(file)
    img_raw.append(image)

In [ ]:
print(img_denoised[0].dtype, img_denoised[0].shape, np.min(img_denoised[0]), np.max(img_denoised[0]))

In [ ]:
# Plot raw and denoised image next to each other

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(img_denoised[0][0])
ax[1].imshow(img_raw[0][0])

## Creating composites of denoised and raw images

In [ ]:
# Create composite images of denoised and raw images

img_composite = []

for i in tqdm(range(len(img_denoised))):
    # Load the corresponding frames from each image series
    image1 = img_denoised[i]
    image2 = img_raw[i]
    
    # Create an empty composite image
    composite_image = np.zeros((100, 2, 1024, 1024), dtype=np.uint16)

    # Assign each image to a separate channel
    composite_image[:, 0, :, :] = image1
    composite_image[:, 1, :, :] = image2

    img_composite.append(composite_image)

img_composite[0].shape

In [ ]:
# Plot single channels of composite image

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(img_composite[0][0][0])
ax[1].imshow(img_composite[0][1][0])

In [188]:
# Initializes Napari viewer

#viewer = napari.Viewer()

In [189]:
# Adds image to Napari viewer

#viewer.add_image(img_composite[0])

In [190]:
#print(img_raw_files[0])

In [ ]:
# Save merged images

for img, tiff_file in zip(img_composite, img_raw_files):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    print(file_name)
    
    # Define the output file path
    output_file = os.path.join(img_composite_dir, file_name + '_merged.tif')
    print(output_file)
    
    # Save the manipulated image as a TIFF
    tf.imwrite(output_file, img, metadata={"axes": "TCYX"}, imagej=True)